# COVID-19 Detection from Chest X-rays using CNN

**Objective.** Build and compare CNN models that classify a chest X-ray into
`Covid`, `Normal`, or `Viral Pneumonia`, so a healthcare provider could get a
fast, automated first read on a scan instead of waiting on manual
radiological review.

**Dataset.** [Kaggle: covid19-image-dataset](https://www.kaggle.com/datasets/pranavraikokte/covid19-image-dataset).
This notebook expects the dataset already unzipped locally at
`data/Covid19-dataset/{train,test}/{Covid,Normal,Viral Pneumonia}/` (the
project ships with it there already — see `README.md` if you're pointing it
at a different copy).

**How this notebook is organized.** Each section below is numbered to match
the project brief's task list 1-9, and opens with a short *why this step
matters* note before the code, so you can see not just *what* each cell does
but *why it's a necessary part of the pipeline* — not just boilerplate.

**Shared code.** Two files are imported throughout instead of re-writing
the same logic inline:
- `preprocessing.py` — the ONE place image resizing/scaling and dataset
  loading happens. `app.py` (the Streamlit app) imports the exact same
  functions, which is what guarantees a prediction made in the app matches
  what the model saw during training (no "train/inference skew").
- `model_utils.py` — the three CNN architectures, plus a save/load helper
  that keeps a weights-only backup in case a `.keras` file trained in one
  TensorFlow version won't load in another.


## Task 0: Setup — import libraries

We need: `os`/`json` for filesystem and metadata handling, `numpy`/`pandas`
for array and tabular work, `matplotlib`/`seaborn` for plots, `PIL` (via
`preprocessing.py`) for image loading, `scikit-learn` for the train/val/test
split and all evaluation metrics, and `tensorflow`/`keras` for the CNNs
themselves. `keras_tuner` is imported later, only inside Task 7, so that the
rest of the notebook still runs even on a machine where it isn't installed.


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_auc_score, precision_recall_fscore_support
)
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping

# Our own shared modules — see the markdown above for why these exist.
from preprocessing import IMG_SIZE, CLASS_NAMES, find_dataset_root, load_dataset_from_folders
from model_utils import build_basic_cnn, build_transfer_model, save_model_robust

%matplotlib inline
sns.set_theme(style="whitegrid")


In [ ]:
# ---- Configuration -----------------------------------------------------
# Kept small so the whole notebook finishes in minutes on a laptop CPU.
# Raise EPOCHS / TUNER_MAX_TRIALS if you have a GPU and want stronger models.
DATASET_CANDIDATES = ["data/Covid19-dataset", "Covid19-dataset", "./Covid19-dataset"]
OUTPUT_DIR = "outputs"
MODEL_DIR = "models"
RANDOM_STATE = 42
VAL_SPLIT = 0.2
EPOCHS = 15
BATCH_SIZE = 16
TUNER_MAX_TRIALS = 5
TUNER_EPOCHS = 6

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)


## Task 1: Data Loading and Exploration

**Why this step matters.** Before writing a single line of model code we
need to know: where the images live, how many there are per class, and what
they look like. Skipping this and jumping straight to modeling is how you
end up debugging a "why is accuracy stuck at 33%" mystery three hours later
that turns out to be a folder-naming typo.

`find_dataset_root()` checks a few likely folder locations so the notebook
doesn't break if you've placed the unzipped dataset somewhere slightly
different. `load_dataset_from_folders()` walks `Covid/`, `Normal/`, and
`Viral Pneumonia/`, resizing every image to `IMG_SIZE` (128x128, set in
`preprocessing.py`) and scaling pixels to `[0, 1]` **as it loads them** —
so by the time this cell finishes, preprocessing (Task 2) is already half done.


In [ ]:
dataset_root = find_dataset_root(DATASET_CANDIDATES)
print(f"Dataset found at: {dataset_root}")

X_train_full, y_train_full, train_paths = load_dataset_from_folders(
    os.path.join(dataset_root, "train"), CLASS_NAMES, IMG_SIZE
)
X_test, y_test, test_paths = load_dataset_from_folders(
    os.path.join(dataset_root, "test"), CLASS_NAMES, IMG_SIZE
)

print(f"Train+val images: {X_train_full.shape[0]}  |  Test images: {X_test.shape[0]}")
print(f"Image tensor shape: {X_train_full.shape[1:]} (already normalized to [0,1])")


**Dataset size per class.** We print this now because it's the first sign
of whether we'll need to handle class imbalance later (Task 6) — a large gap
between class counts here is exactly what would cause a naively-trained
model to just always predict the majority class.


In [ ]:
counts_train = pd.Series(y_train_full).value_counts().sort_index()
counts_test = pd.Series(y_test).value_counts().sort_index()
print("Class counts:")
for idx, name in enumerate(CLASS_NAMES):
    print(f"  {name:<18} train+val={counts_train.get(idx,0):<4} test={counts_test.get(idx,0)}")


## Task 2: Data Preprocessing (continued) — train/validation/test split

**Why we split before doing anything else with the data.** Everything from
here on — EDA aside — needs to be careful not to let information from
validation/test images leak into decisions made about training. We already
normalized pixels and effectively "encoded" labels as integer indices while
loading (index into `CLASS_NAMES`) — that's a deliberate simplification
versus one-hot encoding, because Keras's `sparse_categorical_crossentropy`
loss accepts integer labels directly and saves us an encoding/decoding step.

**Why `stratify=y_train_full`:** with only ~250 training images split three
ways, a plain random split could easily under-represent one class in
validation purely by chance. Stratifying guarantees the same class
proportions in both the training and validation subsets.


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=VAL_SPLIT, stratify=y_train_full, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")


## Task 3: Exploratory Data Analysis (EDA)

**Why this matters.** Two things we look for here tend to explain most of a
model's later mistakes: (1) class imbalance (seen in the bar chart) and
(2) visual similarity between classes — Viral Pneumonia and COVID-19 often
look alike on a chest X-ray to the untrained eye, which is a hint the model
may confuse them too later in the confusion matrix (Task 5).


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=[CLASS_NAMES[i] for i in counts_train.index], y=counts_train.values, ax=ax)
ax.set_title("Training set class distribution")
ax.set_ylabel("Number of images")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "class_distribution.png"), dpi=120)
plt.show()


In [ ]:
fig, axes = plt.subplots(len(CLASS_NAMES), 4, figsize=(12, 3 * len(CLASS_NAMES)))
rng = np.random.default_rng(RANDOM_STATE)
for row, class_idx in enumerate(range(len(CLASS_NAMES))):
    idxs = np.where(y_train_full == class_idx)[0]
    sample_idxs = rng.choice(idxs, size=min(4, len(idxs)), replace=False)
    for col, s_idx in enumerate(sample_idxs):
        axes[row, col].imshow(X_train_full[s_idx])
        axes[row, col].set_title(CLASS_NAMES[class_idx], fontsize=9)
        axes[row, col].axis("off")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "sample_images.png"), dpi=120)
plt.show()


## Task 6: Handle Class Imbalance (computed here, used throughout Task 4)

**Why this section appears here instead of after Task 5:** class weights
are an *argument to model training* (`model.fit(..., class_weight=...)`),
so they have to exist before we can train Models 1-3 below. We're following
the project brief's numbering everywhere else; here we simply compute the
Task 6 artifact early because the pipeline requires it early, and revisit
the *result* (does it help or hurt?) again after evaluation.

**Why `class_weight="balanced"` instead of duplicating minority-class
images (naive oversampling):** with only ~250 training images total,
duplicating exact files risks the model memorizing specific pixels rather
than learning generalizable features. `class_weight` achieves the same
"pay more attention to under-represented classes" effect purely through the
loss function — no duplicate images required. We *also* use augmentation
(Model 3, below) as a complementary, non-duplicating way to effectively grow
the minority classes.


In [ ]:
class_weights_array = compute_class_weight(
    class_weight="balanced", classes=np.unique(y_train), y=y_train
)
class_weight_dict = {i: w for i, w in enumerate(class_weights_array)}
print("Class weights (balanced):",
      {CLASS_NAMES[k]: round(v, 3) for k, v in class_weight_dict.items()})


## Task 4: CNN Model Building — Model 1: Basic CNN

**Why start with a from-scratch CNN at all, when transfer learning
(Models 2-3) usually wins:** this is our *baseline*. Without it, we'd have
no way to tell whether the extra complexity of transfer learning is
actually earning its keep on this dataset, or whether a simple model gets
us most of the way there already.

**Architecture:** `Conv2D -> MaxPooling` repeated three times (learning
increasingly abstract features — edges, then textures, then shapes), then
`Flatten -> Dense -> Dropout -> Dense(softmax)` to turn those features into
a 3-class probability distribution. Dropout (30%) is there specifically
because with only ~250 training images, an unregularized dense layer with
millions of parameters would overfit almost immediately.

We use `EarlyStopping` (patience=4 epochs on validation loss) so training
stops automatically once the model starts overfitting, and automatically
restores the best-performing weights rather than the final epoch's weights.


In [ ]:
model1 = build_basic_cnn(input_shape=X_train.shape[1:])
model1.summary()


In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)

history1 = model1.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=2,
)


## Task 4: CNN Model Building — Model 2: Transfer Learning

**Why transfer learning helps here specifically:** with only ~250 training
images, training a deep CNN from scratch to recognize subtle lung-tissue
patterns is asking a lot. A network pretrained on ImageNet (millions of
photos) has already learned general-purpose visual features — edges,
textures, shapes — that transfer surprisingly well to X-rays, even though
X-rays look nothing like ImageNet's natural photos. We only need to teach
the *last* few layers to specialize for lung pathology.

**Why MobileNetV2 rather than VGG16/ResNet50** (both mentioned as options
in the brief, and both wired up in `model_utils.py` if you want to swap and
compare — just pass `backbone_name="VGG16"` or `"ResNet50"`): MobileNetV2
has ~3.5M parameters vs. VGG16's ~138M. On a dataset this small, a backbone
as large as VGG16 would overfit almost instantly; MobileNetV2 also trains
fast enough to iterate on CPU.

**"Fine-tune the last few layers"** (the brief's wording) means: freeze most
of the pretrained backbone (protecting the general features it already
learned) and only let the last `fine_tune_last_n` layers update during
training, so the top of the network can adapt to X-ray-specific patterns
without destroying everything below it.


In [ ]:
model2 = build_transfer_model(input_shape=X_train.shape[1:], backbone_name="MobileNetV2")
model2.summary()


In [ ]:
history2 = model2.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=2,
)


## Task 4: CNN Model Building — Model 3: Transfer Learning + Data Augmentation

**Why add augmentation on top of Model 2 rather than as a separate,
unrelated model:** this directly answers the brief's Task 6 instruction to
try "oversampling using data augmentation" for class imbalance.
`ImageDataGenerator` generates realistic variations (small rotations,
shifts, zooms, horizontal flips) of every training image, on the fly, every
epoch — increasing the effective diversity of every class (including the
minority ones) without ever duplicating an exact image. Combined with the
`class_weight` we're still passing to `.fit()`, this tackles imbalance from
two independent angles at once.

A **technical note that matters if you re-run this on a different Keras
version:** we deliberately do **not** pass `steps_per_epoch` to `.fit()`
below. `train_datagen.flow(...)` already knows its own length; explicitly
specifying `steps_per_epoch` alongside it caused training to silently stop
consuming batches after the very first epoch when this was tested — a
Keras 3 quirk with `Sequence`-based generators. Leaving `steps_per_epoch`
unset lets Keras infer it correctly every epoch.


In [ ]:
train_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest",
)
train_datagen.fit(X_train)
train_gen = train_datagen.flow(X_train, y_train, batch_size=BATCH_SIZE, seed=RANDOM_STATE)

model3 = build_transfer_model(input_shape=X_train.shape[1:], backbone_name="MobileNetV2")

history3 = model3.fit(
    train_gen,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=2,
)


## Task 5: Model Evaluation

**Why we score all three models with exactly the same function.** If each
model were evaluated with slightly different code, an apparent difference
in performance could just be a measurement artifact rather than a real
difference in model quality. `evaluate_model()` below is called identically
for every model, on the same untouched test set, computing every metric the
brief asks for: accuracy, macro-averaged precision/recall/F1 (macro-average
so the minority classes count equally, not just the majority class),
multi-class ROC-AUC, and a confusion matrix. We also plot train-vs-validation
loss/accuracy curves per model, which is how we check for overfitting — a
model whose training accuracy keeps climbing while validation accuracy
plateaus or drops is memorizing, not learning.


In [ ]:
def evaluate_model(model, X, y_true, name):
    """Compute every required metric for one model on one split, print a
    classification report, save a confusion-matrix plot, and return a dict
    (used to build the Task 8 comparison table)."""
    y_proba = model.predict(X, verbose=0)
    y_pred = np.argmax(y_proba, axis=1)

    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    try:
        roc_auc = roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro")
    except ValueError:
        roc_auc = float("nan")

    cm = confusion_matrix(y_true, y_pred)
    report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0)

    print(f"--- {name} ---")
    print(f"Accuracy: {acc:.4f} | Precision(macro): {precision:.4f} | "
          f"Recall(macro): {recall:.4f} | F1(macro): {f1:.4f} | ROC-AUC(macro): {roc_auc:.4f}")
    print(report)

    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title(f"Confusion matrix - {name}")
    plt.tight_layout()
    safe_name = name.lower().replace(" ", "_")
    plt.savefig(os.path.join(OUTPUT_DIR, f"confusion_matrix_{safe_name}.png"), dpi=120)
    plt.show()

    return {
        "model": name, "test_accuracy": acc, "precision_macro": precision,
        "recall_macro": recall, "f1_macro": f1, "roc_auc_macro": roc_auc,
        "trainable_params": int(np.sum([np.prod(v.shape) for v in model.trainable_weights])),
        "total_params": model.count_params(),
    }


def plot_training_curves(history, name):
    """Task 5 also asks us to check for overfitting via train-vs-val curves."""
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(history.history["loss"], label="train")
    axes[0].plot(history.history["val_loss"], label="val")
    axes[0].set_title(f"{name} - Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
    axes[1].plot(history.history["accuracy"], label="train")
    axes[1].plot(history.history["val_accuracy"], label="val")
    axes[1].set_title(f"{name} - Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
    plt.tight_layout()
    safe_name = name.lower().replace(" ", "_")
    plt.savefig(os.path.join(OUTPUT_DIR, f"training_curves_{safe_name}.png"), dpi=120)
    plt.show()


In [ ]:
results = []
results.append(evaluate_model(model1, X_test, y_test, "Model 1 - Basic CNN"))
plot_training_curves(history1, "Model 1 - Basic CNN")


In [ ]:
results.append(evaluate_model(model2, X_test, y_test, "Model 2 - Transfer Learning"))
plot_training_curves(history2, "Model 2 - Transfer Learning")


In [ ]:
results.append(evaluate_model(model3, X_test, y_test, "Model 3 - Transfer + Augmentation"))
plot_training_curves(history3, "Model 3 - Transfer + Augmentation")


## Task 7: Model Tuning with Keras Tuner

**Why we tune the Basic CNN specifically, not the transfer-learning
models:** hyperparameter tuning searches *architecture-level* choices
(number of filters per layer, dropout rate, learning rate). The
transfer-learning models' architecture is mostly fixed by the pretrained
backbone, so there's far less to meaningfully search there — the from-scratch
CNN is where these choices actually move the needle. This also reuses
`EarlyStopping` (Task 7 explicitly asks for it) inside every trial, so a bad
hyperparameter combination doesn't waste time training to completion.

If `keras_tuner` isn't installed, this cell is skipped gracefully rather
than crashing the rest of the notebook — install it with
`pip install keras-tuner tensorboard` (the tuner needs `tensorboard`
installed even though we never open a TensorBoard dashboard here — it's an
internal dependency of the trial-logging code).


In [ ]:
try:
    import keras_tuner as kt
    tuner_available = True
except ImportError:
    tuner_available = False
    print("keras_tuner not installed - skipping Task 7. "
          "Install with: pip install keras-tuner tensorboard")


In [ ]:
if tuner_available:
    def build_hp_model(hp):
        conv_filters = (
            hp.Choice("filters_1", [16, 32, 64]),
            hp.Choice("filters_2", [32, 64, 128]),
            hp.Choice("filters_3", [64, 128, 256]),
        )
        dense_units = hp.Choice("dense_units", [64, 128, 256])
        dropout_rate = hp.Float("dropout_rate", 0.2, 0.5, step=0.1)
        learning_rate = hp.Choice("learning_rate", [1e-2, 1e-3, 1e-4])
        return build_basic_cnn(
            input_shape=X_train.shape[1:],
            conv_filters=conv_filters,
            dense_units=dense_units,
            dropout_rate=dropout_rate,
            learning_rate=learning_rate,
        )

    tuner = kt.RandomSearch(
        build_hp_model,
        objective="val_accuracy",
        max_trials=TUNER_MAX_TRIALS,
        executions_per_trial=1,
        overwrite=True,
        directory=os.path.join(OUTPUT_DIR, "kt_search"),
        project_name="covid_cnn_tuning",
    )
    tuner.search(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=TUNER_EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weight_dict,
        callbacks=[EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
        verbose=2,
    )

    best_hp = tuner.get_best_hyperparameters(1)[0]
    print("Best hyperparameters found:", best_hp.values)


In [ ]:
if tuner_available:
    tuned_model = tuner.hypermodel.build(best_hp)
    history_tuned = tuned_model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weight_dict,
        callbacks=[early_stop],
        verbose=2,
    )
    tuned_result = evaluate_model(tuned_model, X_test, y_test, "Model 4 - Tuned Basic CNN")
    plot_training_curves(history_tuned, "Model 4 - Tuned Basic CNN")
    results.append(tuned_result)

    all_models = {
        "Model 1 - Basic CNN": model1,
        "Model 2 - Transfer Learning": model2,
        "Model 3 - Transfer + Augmentation": model3,
        "Model 4 - Tuned Basic CNN": tuned_model,
    }
else:
    all_models = {
        "Model 1 - Basic CNN": model1,
        "Model 2 - Transfer Learning": model2,
        "Model 3 - Transfer + Augmentation": model3,
    }


## Task 8: Model Comparison Table

**Why a single sortable table, not just four separate printouts above:**
side-by-side is the only way to make an actual decision about which model
to ship. Sorting by test accuracy also directly drives Task 9 below — the
top row is automatically the model we save for the Streamlit app.


In [ ]:
comparison_df = pd.DataFrame(results).sort_values("test_accuracy", ascending=False)
comparison_df.to_csv(os.path.join(OUTPUT_DIR, "model_comparison.csv"), index=False)
comparison_df


## Task 9: Save the Best Model (for the Streamlit App)

**Why we save two formats, not just one:** `model.save(...)` (the native
`.keras` format) is the normal way to reload a full model later, but Keras
save-format compatibility occasionally breaks across TensorFlow/Keras
versions — a model saved in one environment (say, this notebook's) can
sometimes fail to load in another (say, the machine `app.py` runs on,
if its TensorFlow version differs). `save_model_robust()` also saves the raw
weights plus enough metadata to rebuild the exact same architecture in code,
so `app.py` can automatically fall back to "rebuild architecture + load
weights only" if the native `.keras` load fails. See `model_utils.py` for
`load_model_robust()`, the counterpart used by the app.

Run `streamlit run app.py` after this cell finishes — the app looks for the
`models/best_model.keras` and `models/metadata.json` files this cell creates.


In [ ]:
best_row = comparison_df.iloc[0]
best_model_name = best_row["model"]
best_model = all_models[best_model_name]
print(f"Best model by test accuracy: {best_model_name} (accuracy={best_row['test_accuracy']:.4f})")

keras_path, weights_path = save_model_robust(best_model, MODEL_DIR, "best_model")

metadata = {
    "best_model_name": best_model_name,
    "img_size": list(IMG_SIZE),
    "class_names": CLASS_NAMES,
    "test_accuracy": float(best_row["test_accuracy"]),
    "is_transfer_model": "Transfer" in best_model_name,
}
with open(os.path.join(MODEL_DIR, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved model to: {keras_path}")
print(f"Saved weights fallback to: {weights_path}")
print("Saved metadata to:", os.path.join(MODEL_DIR, "metadata.json"))
print("\nDONE. In a terminal, in this project folder, run:  streamlit run app.py")


## Next step

Open a terminal in this project folder and run:

```
streamlit run app.py
```

The app loads `models/best_model.keras`, and uses the exact same
`load_and_preprocess_image()` function from `preprocessing.py` that this
notebook uses internally — so any chest X-ray you upload there is treated
identically to how the training images above were treated, and the
confidence scores you see in the app are a faithful match to this
notebook's test-set results.
